# 04 - Pressure ratings and rankings
Rating (runs-equivalent per 100 balls vs an average player in the same state) =
  alpha : ordinary-ball skill (lots of data, reliable)  +  delta : pressure deviation from own baseline (shrunk hard)
Runs and wicket components are shrunk separately, each by its own measured reliability (empirical Bayes).
Outputs: outputs/batter_ratings.csv, outputs/bowler_ratings.csv

In [1]:
import warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 80)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
OUT = ROOT / 'outputs'
Bt = pd.read_parquet(OUT / 'batter_ball_values.parquet')
Bw = pd.read_parquet(OUT / 'bowler_ball_values.parquet')
b = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=['striker_id', 'striker_name', 'bowler_id', 'bowler_name'])
names = pd.concat([b[['striker_id', 'striker_name']].set_axis(['pid', 'name'], axis=1),
                   b[['bowler_id', 'bowler_name']].set_axis(['pid', 'name'], axis=1)]).drop_duplicates('pid').set_index('pid').name

PRESSURE = 80                 # percentile (within tier) that counts as pressure
MIN_BALLS, MIN_INN = 60, 8    # eligibility for the top-10 lists (pressure balls / pressure innings with >=5 balls)
W_LEVEL, W_HIT = 0.90, 0.10   # consistency (hit rate) has ~0 measured reliability, so it carries a small weight

## Empirical-Bayes helpers

In [2]:
def eb_center(m, n, sig2, min_n):
    """Shrink player means m (on n obs) toward the pooled mean. Returns shrunk mean, shrink factor B, tau (true spread)."""
    ok = n >= min_n
    tau2 = max(np.var(m[ok]) - np.mean(sig2 / n[ok]), 0.0)
    mu = np.average(m, weights=n)
    B = tau2 / (tau2 + sig2 / n)
    return mu + B * (m - mu), B, np.sqrt(tau2)


def fit_players(df, pid, pcol, thr=PRESSURE, verbose=False):
    df = df[df[pcol].notna()].copy()
    df['hp'] = df[pcol] >= thr
    df['wr'], df['wo'] = df.vr * df.w, df.vo * df.w
    agg = lambda s: s.groupby(pid).agg(n=('w', 'sum'), raw=('w', 'size'), sr=('wr', 'sum'), so=('wo', 'sum'))
    P, O = agg(df[df.hp]), agg(df[~df.hp])
    T = P.join(O, lsuffix='_p', rsuffix='_o', how='left')
    T['n_o'] = T.n_o.fillna(0).clip(lower=1e-3)
    T[['sr_o', 'so_o', 'raw_o']] = T[['sr_o', 'so_o', 'raw_o']].fillna(0)
    alpha = delta = 0
    var_a = var_d = 0
    info = {}
    for k, (sp, so_) in {'r': ('sr_p', 'sr_o'), 'o': ('so_p', 'so_o')}.items():
        sig2 = df['v' + k].var()
        mp, mo = T[sp] / T.n_p, T[so_] / T.n_o
        mo = mo.where(T.n_o > 1, np.average(mo, weights=T.n_o))
        a, Ba, tau_a = eb_center(mo.values, T.n_o.values, sig2, 100)
        d = (mp - mo).values
        noise = sig2 / T.n_p.values + sig2 / T.n_o.values
        okk = (T.n_p.values >= 30) & (T.n_o.values >= 30)
        tau2_d = max(np.var(d[okk]) - np.mean(noise[okk]), 0.0)
        mu_d = np.average(d, weights=1 / noise)
        Bd = tau2_d / (tau2_d + noise)
        dd = mu_d + Bd * (d - mu_d)
        T['alpha_' + k], T['delta_' + k] = a, dd
        alpha, delta = alpha + a, delta + dd
        var_a = var_a + Ba * sig2 / T.n_o.values
        var_d = var_d + Bd * noise
        info[k] = (tau_a * 100, np.sqrt(tau2_d) * 100)
        T['raw_p_' + k] = mp * 100
    T['alpha'], T['delta'] = alpha, delta
    T['pl'] = alpha + delta
    T['pl_se'] = np.sqrt(var_a + var_d)
    T['delta_se'] = np.sqrt(var_d)
    for c in ['alpha', 'delta', 'pl', 'pl_se', 'delta_se']:
        T[c] *= 100
    T['m_p'] = (T.raw_p_r + T.raw_p_o)
    T['ci_lo'], T['ci_hi'] = T.pl - 1.96 * T.pl_se, T.pl + 1.96 * T.pl_se
    # consistency: share of pressure innings (>=5 pressure balls) with a positive total value
    h = df[df.hp].assign(wv=lambda x: (x.vr + x.vo) * x.w).groupby([pid, 'match_id', 'innings']).agg(n=('wv', 'size'), s=('wv', 'sum')).reset_index()
    h = h[h.n >= 5]
    hh = h.groupby(pid).agg(inn_p=('s', 'size'), hits=('s', lambda s: (s > 0).sum()))
    T = T.join(hh).fillna({'inn_p': 0, 'hits': 0})
    p0 = (h.s > 0).mean()
    T['hit'] = (T.hits + 6 * p0) / (T.inn_p + 6)
    mp_t, mo_t = (T.sr_p + T.so_p) / T.n_p, (T.sr_o + T.so_o) / T.n_o
    T['gap_raw'] = (mp_t - mo_t) * 100
    T['gap_z'] = (mp_t - mo_t) / np.sqrt(df.v.var() * (1 / T.n_p + 1 / T.n_o))
    T['weak_share'] = df[df.hp & df.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])].groupby(pid).size().reindex(T.index).fillna(0) / T.raw_p
    if verbose:
        print('  true player spread (runs/100 balls)  runs: alpha %.1f, delta %.1f | wickets: alpha %.1f, delta %.1f' % (*info['r'], *info['o']))
    T['eligible'] = (T.raw_p >= MIN_BALLS) & (T.inn_p >= MIN_INN)
    return T


def score(T, wl=W_LEVEL, wh=W_HIT, el=None):
    e = T.eligible if el is None else el
    z = lambda s: (s - s[e].mean()) / s[e].std()
    s = wl * z(T.pl) + wh * z(T.hit)
    return s.where(e)


def finish(T):
    T['score'] = score(T)
    T['rank'] = T.score.rank(ascending=False)
    T['pressure_type'] = np.select([T.gap_z >= 1.5, T.gap_z <= -1.5], ['Thrives', 'Dips'], 'Neutral')   # indicative only
    T.insert(0, 'name', T.index.map(names))
    return T.sort_values('score', ascending=False)


print('BATTERS'); BAT = finish(fit_players(Bt, 'striker_id', 'p_bat', verbose=True))
print('BOWLERS'); BOWL = finish(fit_players(Bw, 'bowler_id', 'p_bowl', verbose=True))
print('eligible batters:', int(BAT.eligible.sum()), '| eligible bowlers:', int(BOWL.eligible.sum()))

SHOW = ['name', 'raw_p', 'inn_p', 'm_p', 'alpha', 'delta', 'pl', 'ci_lo', 'ci_hi', 'hit', 'score', 'weak_share']
print('\nTOP 15 BATTERS UNDER PRESSURE (runs-equivalent per 100 balls vs an average player in the same state)')
print(BAT[BAT.eligible].head(15)[SHOW].round(2).to_string())
print('\nTOP 15 BOWLERS UNDER PRESSURE')
print(BOWL[BOWL.eligible].head(15)[SHOW].round(2).to_string())

BATTERS
  true player spread (runs/100 balls)  runs: alpha 11.8, delta 0.0 | wickets: alpha 0.0, delta 0.0
BOWLERS
  true player spread (runs/100 balls)  runs: alpha 6.7, delta 13.3 | wickets: alpha 1.3, delta 0.0
eligible batters: 124 | eligible bowlers: 135

TOP 15 BATTERS UNDER PRESSURE (runs-equivalent per 100 balls vs an average player in the same state)
                        name  raw_p  inn_p    m_p  alpha  delta     pl  ci_lo  ci_hi   hit  score  weak_share
striker_id                                                                                                   
6281         Abhishek Sharma    245   17.0  38.75  31.09   2.82  33.91  26.72  41.09  0.71   3.20        0.03
25669          Priyansh Arya     62    8.0 -19.30  25.31   2.82  28.13  13.99  42.27  0.67   2.55        0.08
6396               Tim David    277   22.0  35.54  22.08   2.82  24.90  14.74  35.06  0.69   2.23        0.00
1318           Andre Russell    166   13.0  23.48  22.04   2.82  24.86  12.82  36.90  0.

## Reliability: do the ratings persist? (odd vs even matches; players with >=30 pressure balls in both halves)

In [3]:
def rel(df, pid, pcol):
    Ta = fit_players(df[df.match_id % 2 == 0], pid, pcol)
    Tb = fit_players(df[df.match_id % 2 == 1], pid, pcol)
    j = Ta.join(Tb, lsuffix='_a', rsuffix='_b', how='inner')
    j = j[(j.raw_p_a >= 30) & (j.raw_p_b >= 30)]
    r = {c: round(spearmanr(j[c + '_a'], j[c + '_b'])[0], 2) for c in ['m_p', 'pl', 'alpha', 'gap_raw', 'hit']}
    r['n_players'] = len(j)
    return r


print('Split-half rank correlation (raw pressure mean = what a naive metric would use)')
print('batters :', rel(Bt, 'striker_id', 'p_bat'))
print('bowlers :', rel(Bw, 'bowler_id', 'p_bowl'))

Split-half rank correlation (raw pressure mean = what a naive metric would use)
batters : {'m_p': np.float64(-0.14), 'pl': np.float64(0.4), 'alpha': np.float64(0.38), 'gap_raw': np.float64(-0.24), 'hit': np.float64(-0.01), 'n_players': 143}
bowlers : {'m_p': np.float64(0.13), 'pl': np.float64(0.31), 'alpha': np.float64(0.35), 'gap_raw': np.float64(0.05), 'hit': np.float64(0.21), 'n_players': 144}


## Sensitivity of the top 10 to our choices

In [4]:
for lab, T, df, pid, pc in [('BATTERS', BAT, Bt, 'striker_id', 'p_bat'), ('BOWLERS', BOWL, Bw, 'bowler_id', 'p_bowl')]:
    base = list(T[T.eligible].head(10).index)
    print(lab, '- overlap with chosen top 10:')
    for w in [(1, 0), (0.6, 0.4), (0.9, 0.1)]:
        print('  weights level/consistency', w, '->', len(set(base) & set(score(T, *w).sort_values(ascending=False).head(10).index)), '/10')
    for thr in (70, 90):
        T2 = fit_players(df, pid, pc, thr=thr)
        print('  pressure threshold', thr, '->', len(set(base) & set(score(T2).sort_values(ascending=False).head(10).index)), '/10')
    d2 = df.assign(w=1.0)
    T3 = fit_players(d2, pid, pc)
    print('  all tiers weighted equally ->', len(set(base) & set(score(T3).sort_values(ascending=False).head(10).index)), '/10')
    d4 = df[~df.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])]
    T4 = fit_players(d4, pid, pc)
    print('  IPL + full-member T20I only ->', len(set(base) & set(score(T4).sort_values(ascending=False).head(10).index)), '/10')
    e = T.eligible & (T.raw_p >= 100) & (T.inn_p >= 12)
    print('  stricter eligibility (100 balls, 12 inns; %d players) ->' % e.sum(), len(set(base) & set(score(T, el=e).sort_values(ascending=False).head(10).index)), '/10')

BATTERS - overlap with chosen top 10:
  weights level/consistency (1, 0) -> 9 /10
  weights level/consistency (0.6, 0.4) -> 7 /10
  weights level/consistency (0.9, 0.1) -> 10 /10
  pressure threshold 70 -> 8 /10
  pressure threshold 90 -> 3 /10
  all tiers weighted equally -> 10 /10
  IPL + full-member T20I only -> 9 /10
  stricter eligibility (100 balls, 12 inns; 66 players) -> 8 /10
BOWLERS - overlap with chosen top 10:
  weights level/consistency (1, 0) -> 9 /10
  weights level/consistency (0.6, 0.4) -> 8 /10
  weights level/consistency (0.9, 0.1) -> 10 /10
  pressure threshold 70 -> 7 /10
  pressure threshold 90 -> 6 /10
  all tiers weighted equally -> 10 /10
  IPL + full-member T20I only -> 10 /10
  stricter eligibility (100 balls, 12 inns; 87 players) -> 10 /10


## Consensus ranking: average the score over 12 reasonable specifications
pressure threshold {70, 80, 90} x weak-tier weight {0.5, 1.0} x consistency weight {0.1, 0}

In [5]:
def consensus(df, pid, pc, base, min_balls=50):
    el = base.eligible                       # z-scores are calibrated on players meeting the full threshold ...
    ok = base.raw_p >= min_balls             # ... but every player with >= 50 pressure balls gets a score (thin samples are flagged)
    zs = []
    weak = df.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])
    for thr in (70, 80, 90):
        for tw in (0.5, 1.0):
            T = fit_players(df.assign(w=df.w.where(~weak, tw)), pid, pc, thr=thr).reindex(base.index)
            z = lambda x: (x - x[el].mean()) / x[el].std()
            for wl, wh in [(0.9, 0.1), (1.0, 0.0)]:
                zs.append((wl * z(T.pl) + wh * z(T.hit)).where(ok))
    return pd.concat(zs, axis=1).mean(axis=1)


for lab, T, df, pid, pc in [('BAT', BAT, Bt, 'striker_id', 'p_bat'), ('BOWL', BOWL, Bw, 'bowler_id', 'p_bowl')]:
    T['consensus'] = consensus(df, pid, pc, T)
    T['final_rank'] = T.consensus.where(T.eligible).rank(ascending=False)
    print(lab, 'overlap of consensus top 10 with base-spec top 10:', len(set(T.nsmallest(10, 'final_rank').index) & set(T.nsmallest(10, 'rank').index)), '/10')

FIN = ['name', 'final_rank', 'rank', 'raw_p', 'inn_p', 'm_p', 'alpha', 'pl', 'ci_lo', 'ci_hi', 'hit', 'consensus', 'pressure_type']
print('\nFINAL TOP 10 BATTERS'); print(BAT.nsmallest(10, 'final_rank')[FIN].round(2).to_string())
print('\nFINAL TOP 10 BOWLERS'); print(BOWL.nsmallest(10, 'final_rank')[FIN].round(2).to_string())

BAT overlap of consensus top 10 with base-spec top 10: 8 /10
BOWL overlap of consensus top 10 with base-spec top 10: 7 /10

FINAL TOP 10 BATTERS
                        name  final_rank  rank  raw_p  inn_p    m_p  alpha     pl  ci_lo  ci_hi   hit  consensus pressure_type
striker_id                                                                                                                    
6281         Abhishek Sharma         1.0   1.0    245   17.0  38.75  31.09  33.91  26.72  41.09  0.71       3.42       Neutral
25669          Priyansh Arya         2.0   2.0     62    8.0 -19.30  25.31  28.13  13.99  42.27  0.67       2.51       Neutral
6396               Tim David         3.0   3.0    277   22.0  35.54  22.08  24.90  14.74  35.06  0.69       2.29       Neutral
1318           Andre Russell         4.0   4.0    166   13.0  23.48  22.04  24.86  12.82  36.90  0.65       2.21       Neutral
1699        Suryakumar Yadav         5.0   5.0    382   25.0  26.82  19.59  22.41  15.67  29.

## Pressure gap vs own baseline (raw, indicative only: the gap itself has low reliability)

In [6]:
for lab, T in [('BATTERS', BAT), ('BOWLERS', BOWL)]:
    e = T[T.eligible]
    print('\n%s: pressure_type counts' % lab, e.pressure_type.value_counts().to_dict())
    print(' biggest risers:'); print(e.sort_values('gap_z', ascending=False).head(4)[['name', 'alpha', 'gap_raw', 'gap_z', 'raw_p']].round(2).to_string())
    print(' biggest dippers:'); print(e.sort_values('gap_z').head(4)[['name', 'alpha', 'gap_raw', 'gap_z', 'raw_p']].round(2).to_string())

BAT.to_csv(OUT / 'batter_ratings.csv')
BOWL.to_csv(OUT / 'bowler_ratings.csv')
print('\nsaved ratings')


BATTERS: pressure_type counts {'Neutral': 111, 'Thrives': 8, 'Dips': 5}
 biggest risers:
                        name  alpha  gap_raw  gap_z  raw_p
striker_id                                                
4260        Heinrich Klaasen   7.77    54.14   3.23    287
109          Wriddhiman Saha -18.47    70.21   2.56    102
6460            Ayush Badoni -10.59    43.00   2.28    265
1107           Mohammad Nabi  -5.55    41.32   2.07    314
 biggest dippers:
                       name  alpha  gap_raw  gap_z  raw_p
striker_id                                               
2101        Quinton de Kock   3.29   -63.10  -2.90    155
2367            Travis Head  18.00   -44.38  -2.06    169
2416            Kyle Mayers   5.34   -47.52  -1.86    118
3639           Brandon King  -3.22   -40.67  -1.55    103

BOWLERS: pressure_type counts {'Neutral': 121, 'Thrives': 7, 'Dips': 7}
 biggest risers:
                          name  alpha  gap_raw  gap_z  raw_p
bowler_id                              